In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "notebook_setup.py").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_setup import get_dataset_path

import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ==========================
# ==========================
mat_path = get_dataset_path("CETL_data_processed.mat")
data = sio.loadmat(mat_path)

processed_Qd_mat = data["processed_Qd_mat"]   # 201 x 3511

Qmax_all = processed_Qd_mat[-1, :].astype(float).ravel()
n_samples = Qmax_all.size

C0 = 15.5  # Ah
SOH_all = Qmax_all / C0 * 100.0

# ==========================
# ==========================
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20

fig_width1 = 6
fig_height1 = 4
fig_width2 = 8
fig_height2 = 5

# ==========================
# ==========================
qmin, qmax = np.min(Qmax_all), np.max(Qmax_all)
iqr = np.percentile(Qmax_all, 75) - np.percentile(Qmax_all, 25)

if iqr > 0:
    bin_width = 2 * iqr / np.cbrt(n_samples)
    if bin_width <= 0:
        bin_width = (qmax - qmin) / 20.0
else:
    bin_width = (qmax - qmin) / 20.0 if qmax > qmin else 0.1

if bin_width <= 0:
    bin_width = 0.1

bins = np.arange(qmin, qmax + bin_width, bin_width)
if bins.size < 5:
    bins = np.linspace(qmin, qmax, 10)

counts, edges = np.histogram(Qmax_all, bins=bins)
bin_centres = 0.5 * (edges[:-1] + edges[1:])
bin_widths = np.diff(edges)

bin_std = np.full_like(bin_centres, np.nan, dtype=float)
for i in range(len(edges) - 1):
    left, right = edges[i], edges[i + 1]
    if i < len(edges) - 2:
        mask = (Qmax_all >= left) & (Qmax_all < right)
    else:
        mask = (Qmax_all >= left) & (Qmax_all <= right)
    if np.any(mask):
        bin_std[i] = np.std(Qmax_all[mask], ddof=1)

fig1, ax1 = plt.subplots(figsize=(fig_width1, fig_height1))

bars = ax1.bar(
    bin_centres,
    counts,
    width=bin_widths,
    align="center",
    color="#a6cee3",
    edgecolor="black",
    linewidth=0.8,
)

ax1.set_xlabel(r"Maximum capacity (Ah)", labelpad=10)
ax1.set_ylabel("Cell number", labelpad=10)
ax1.grid(False)

for spine in ["top", "right"]:
    ax1.spines[spine].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(
    bin_centres,
    bin_std,
    color="black",
    marker="o",
    linewidth=1.5,
    markersize=4,
    label=r"Intra-bin SD",
)
ax2.fill_between(
    bin_centres,
    bin_std,
    step="mid",
    alpha=0.15,
    color="grey",
)

ax2.set_ylabel(r"Intra-bin SD (Ah)", labelpad=15)
ax2.spines["top"].set_visible(False)
ax2.legend(loc="upper right", frameon=False)

fig1.tight_layout()
# fig1.savefig("Qmax_distribution_with_std.png", dpi=300, bbox_inches="tight")

# ==========================
# ==========================

soh_bins = np.arange(0, 100, 3.0)  # 0,3,6,...,99
soh_bins = np.append(soh_bins, 100.0)

soh_counts = []
soh_labels = []

for i in range(len(soh_bins) - 1):
    l, r = soh_bins[i], soh_bins[i + 1]
    if i < len(soh_bins) - 2:
        mask = (SOH_all >= l) & (SOH_all < r)
    else:
        mask = (SOH_all >= l) & (SOH_all <= r)

    cnt = int(np.sum(mask))
    if cnt > 0:
        soh_counts.append(cnt)
        soh_labels.append(f"{l:.0f}–{r:.0f}%")

soh_counts = np.array(soh_counts, dtype=float)
total = soh_counts.sum()
percentages = soh_counts / total * 100.0

cmap_pie = plt.get_cmap("tab20c")
colors = cmap_pie(np.linspace(0.1, 0.9, len(soh_counts)))

def add_3d_wedge(ax, theta1_deg, theta2_deg, r=1.0, z=0.0, height=0.25, color="C0"):
    """Add a 3D pie-slice column and return its top collection for the legend."""
    theta = np.linspace(np.deg2rad(theta1_deg), np.deg2rad(theta2_deg), 40)

    x_outer_top = r * np.cos(theta)
    y_outer_top = r * np.sin(theta)
    z_top = np.full_like(theta, z + height)

    x_outer_bottom = r * np.cos(theta)
    y_outer_bottom = r * np.sin(theta)
    z_bottom = np.full_like(theta, z)

    verts_top = [(0, 0, z + height)] + list(zip(x_outer_top, y_outer_top, z_top))
    poly_top = Poly3DCollection([verts_top], facecolor=color, edgecolor="k", linewidths=0.5)
    ax.add_collection3d(poly_top)

    verts_side = list(zip(x_outer_bottom, y_outer_bottom, z_bottom)) + \
                 list(zip(x_outer_top[::-1], y_outer_top[::-1], z_top[::-1]))
    poly_side = Poly3DCollection([verts_side], facecolor=color, edgecolor="k", linewidths=0.3, alpha=0.95)
    ax.add_collection3d(poly_side)

    return poly_top

fig2 = plt.figure(figsize=(fig_width2, fig_height2))
ax3 = fig2.add_subplot(111, projection="3d")

ax3.set_position([0.02, 0.05, 0.75, 0.9])

start_angle = 0.0
r = 1.1
h = 0.25

handles = []
legend_labels = []

for size, pct, label, color in zip(soh_counts, percentages, soh_labels, colors):
    frac = size / total
    angle = 360.0 * frac
    theta1 = start_angle
    theta2 = start_angle + angle

    handle = add_3d_wedge(ax3, theta1, theta2, r=r, z=0.0, height=h, color=color)
    handles.append(handle)
    legend_labels.append(f"{label} ({pct:.1f}%)")

    start_angle += angle

ax3.view_init(elev=30, azim=-60)
ax3.set_xlim(-1.6, 1.6)
ax3.set_ylim(-1.6, 1.6)
ax3.set_zlim(0, 0.5)
ax3.set_box_aspect((1, 1, 0.2))
ax3.set_axis_off()

leg = ax3.legend(
    handles,
    legend_labels,
    title="SOH ranges",
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    borderaxespad=0.5,
    frameon=False,
    ncol=2,
    columnspacing=0.8,
    handlelength=1.2,
)
plt.setp(leg.get_title(), fontsize=20)


plt.show()

fig1.savefig('cap_distribution.png', dpi=300, bbox_inches='tight')
fig2.savefig('soh_distribution.png', dpi=300, bbox_inches='tight')

In [ ]:
fig2 = plt.figure(figsize=(fig_width, fig_height))
ax3 = fig2.add_subplot(111, projection="3d")

ax3.set_position([0.02, 0.05, 0.75, 0.9])

start_angle = 0.0
r = 1.1
h = 0.25

handles = []
legend_labels = []

for size, pct, label, color in zip(soh_counts, percentages, soh_labels, colors):
    frac = size / total
    angle = 360.0 * frac
    theta1 = start_angle
    theta2 = start_angle + angle

    handle = add_3d_wedge(ax3, theta1, theta2, r=r, z=0.0, height=h, color=color)
    handles.append(handle)
    legend_labels.append(f"{label} ({pct:.1f}%)")

    start_angle += angle

ax3.view_init(elev=30, azim=-60)
ax3.set_xlim(-1.6, 1.6)
ax3.set_ylim(-1.6, 1.6)
ax3.set_zlim(0, 0.5)
ax3.set_box_aspect((1, 1, 0.35))
ax3.set_axis_off()

leg = ax3.legend(
    handles,
    legend_labels,
    title="SOH ranges",
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    borderaxespad=0.5,
    frameon=False,
    ncol=2,
    columnspacing=0.8,
    handlelength=1.2,
)
plt.setp(leg.get_title(), fontsize=18)


plt.show()
